##Build Races Dimensions

1. Read silver races table
2. Read silver circuits table
3. Join the data from races with circuits using circuit_id
4. Select the required columns
    - races.session
    - reces.round
    - races.race_name
    - races.race_date
    - circuits.circuit_name
    - circuits.locality
    __- circuits.country
5. Write the transformed data to gold dim_races table.

Below changes are required to implement Incremental Load Processing
1. Accept batch_id as a parameter to the notebook
2. Process data for only the batch_id being passed in (i.e., filter reading from silver using the batch_id)
3. Add created_timestamp, updated_timestamp to the gold table.
4. Merge the processed data to the gold table
     - created_timestamp should only be populated at the time of inserting/ creating the record. It should not be updated during the merge update.

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"

#### Step 1: Read silver table : circuits, races

In [0]:
circuits_df = (
    spark.table(f"{catalog_name}.{silver_schema}.circuits")
        .filter(F.col("batch_id") == v_batch_id)
)

races_df = (
    spark.table(f"{catalog_name}.{silver_schema}.races")
        .filter(F.col("batch_id") == v_batch_id)
)

#### Step 2: Join races with circuits using circuit_id

select the following columns:
1. races.season
2. races.round
3. races.race_name
4. races.race_date
5. circuits.circuit_name
6. circuits.locality
7. circuits.country

In [0]:
dim_races_df =(
            races_df
                .join(
                    circuits_df, 
                    races_df.circuit_id == circuits_df.circuit_id,
                    "inner"
                    )
                .select(
                    races_df.season,
                    races_df.round,
                    races_df.race_name,
                    races_df.race_date,
                    circuits_df.circuit_name,
                    circuits_df.locality,
                    circuits_df.country
                )
        )

In [0]:
display(dim_races_df)

season,round,race_name,race_date,circuit_name,locality,country
1950,3,Indianapolis 500,1950-05-30,Indianapolis Motor Speedway,Indianapolis,USA
1950,5,Belgian Grand Prix,1950-06-18,Circuit De Spa-francorchamps,Spa,Belgium
1951,8,Spanish Grand Prix,1951-10-28,Circuit De Pedralbes,Barcelona,Spain
1952,4,French Grand Prix,1952-07-06,Rouen-les-essarts,Rouen,France
1954,6,German Grand Prix,1954-08-01,Nürburgring,Nürburg,Germany
1957,2,Monaco Grand Prix,1957-05-19,Circuit De Monaco,Monte Carlo,Monaco
1957,8,Italian Grand Prix,1957-09-08,Autodromo Nazionale Di Monza,Monza,Italy
1961,7,Italian Grand Prix,1961-09-10,Autodromo Nazionale Di Monza,Monza,Italy
1962,8,United States Grand Prix,1962-10-07,Watkins Glen,New York State,USA
1963,4,French Grand Prix,1963-06-30,Reims-gueux,Reims,France


####Step 3: Write the transformed data to the gold dim_races table

In [0]:
write_to_gold(
    input_df = dim_races_df,
    target_table = target_table,
    merge_condition="t.season=s.season AND t.round=s.round",
    columns_to_update=[
        "race_name",
        "race_date",
        "circuit_name",
        "locality",
        "country"
    ]
)

In [0]:
display(spark.table(target_table))

season,round,race_name,race_date,circuit_name,locality,country,created_timestamp,updated_timestamp
1950,3,Indianapolis 500,1950-05-30,Indianapolis Motor Speedway,Indianapolis,USA,2026-08-05T15:41:00.153Z,2026-08-05T15:41:16.240Z
1950,5,Belgian Grand Prix,1950-06-18,Circuit De Spa-francorchamps,Spa,Belgium,2026-08-05T15:41:00.153Z,2026-08-05T15:41:16.240Z
1951,8,Spanish Grand Prix,1951-10-28,Circuit De Pedralbes,Barcelona,Spain,2026-08-05T15:41:00.153Z,2026-08-05T15:41:16.240Z
1952,4,French Grand Prix,1952-07-06,Rouen-les-essarts,Rouen,France,2026-08-05T15:41:00.153Z,2026-08-05T15:41:16.240Z
1954,6,German Grand Prix,1954-08-01,Nürburgring,Nürburg,Germany,2026-08-05T15:41:00.153Z,2026-08-05T15:41:16.240Z
1957,2,Monaco Grand Prix,1957-05-19,Circuit De Monaco,Monte Carlo,Monaco,2026-08-05T15:41:00.153Z,2026-08-05T15:41:16.240Z
1957,8,Italian Grand Prix,1957-09-08,Autodromo Nazionale Di Monza,Monza,Italy,2026-08-05T15:41:00.153Z,2026-08-05T15:41:16.240Z
1961,7,Italian Grand Prix,1961-09-10,Autodromo Nazionale Di Monza,Monza,Italy,2026-08-05T15:41:00.153Z,2026-08-05T15:41:16.240Z
1962,8,United States Grand Prix,1962-10-07,Watkins Glen,New York State,USA,2026-08-05T15:41:00.153Z,2026-08-05T15:41:16.240Z
1963,4,French Grand Prix,1963-06-30,Reims-gueux,Reims,France,2026-08-05T15:41:00.153Z,2026-08-05T15:41:16.240Z
